In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# -----------------------------
# 1. Parameters
# -----------------------------
vocab_size = 10000
max_len = 200
embedding_dim = 128
hidden_dim = 128
batch_size = 64
epochs = 2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# 2. Load Dataset
# -----------------------------
print("Loading IMDB dataset...")
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=vocab_size)

# -----------------------------
# 3. Preprocess Data
# -----------------------------
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

x_train = torch.tensor(x_train, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=batch_size)

# -----------------------------
# 4. Define LSTM Model
# -----------------------------
class LSTMModel(nn.Module):
    def __init__(self):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        x = self.embedding(x)
        _, (hidden, _) = self.lstm(x)
        out = self.fc(hidden[-1])
        return out

model = LSTMModel().to(device)

# -----------------------------
# 5. Loss and Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------------
# 6. Training
# -----------------------------
print("Training started...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

print("Training finished!")

# -----------------------------
# 7. Evaluation
# -----------------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = correct / total
print("Accuracy:", accuracy)

# -----------------------------
# 8. Predict Custom Text
# -----------------------------
word_index = imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

def encode_review(text):
    tokens = text.lower().split()
    encoded = []
    for word in tokens:
        if word in word_index and word_index[word] < vocab_size:
            encoded.append(word_index[word])
        else:
            encoded.append(0)
    padded = pad_sequences([encoded], maxlen=max_len)
    return torch.tensor(padded, dtype=torch.long)

def predict_sentiment(text):
    model.eval()
    encoded = encode_review(text).to(device)
    with torch.no_grad():
        output = model(encoded)
        _, prediction = torch.max(output, 1)
    return "Positive 😊" if prediction.item() == 1 else "Negative 😔"

# Test
review1 = "This movie was incredibly awesome and I loved it"
review2 = "The movie was boring slow and terrible"

print("Review:", review1)
print("Sentiment:", predict_sentiment(review1))

print("Review:", review2)
print("Sentiment:", predict_sentiment(review2))


Loading IMDB dataset...
17464789/17464789 [==============================] - 5s 0us/step
Training started...
Epoch 1/2, Loss: 227.9376
Epoch 2/2, Loss: 193.2776
Training finished!
Accuracy: 0.78176
1641221/1641221 [==============================] - 0s 0us/step
Review: This movie was incredibly awesome and I loved it
Sentiment: Negative 😔
Review: The movie was boring slow and terrible
Sentiment: Negative 😔
